# TrafficLens: YOLO 기반 도로 위 차량·보행자 실시간 인식 시스템

**팀 로드비전** · 팀장 장창근 | 팀원 반광수 · 지명준 · 장혜원 · MBC 아카데미

YOLOv8로 도로 영상 속 차량과 보행자를 찾고, ByteTrack으로 추적하고, 화면에 그은 기준선을 지나간 수를 셉니다.
실험으로 문제점을 찾아내고, 개선 전후를 숫자와 그래프로 비교합니다.

### 학습 목표
1. 사전 학습된 YOLO 모델로 원하는 클래스만 골라 탐지할 수 있다.
2. 객체 추적(ByteTrack)과 기준선 통과 판정으로 교통량을 셀 수 있다.
3. 모델 크기·입력 해상도가 정확도와 속도에 주는 영향을 실험으로 확인할 수 있다.
4. 실험에서 발견한 오류를 분석하고 개선 방법을 적용·검증할 수 있다.

### 목차
| 단계 | 내용 |
| --- | --- |
| 0 | 배경 이론: YOLO, ByteTrack, 기준선 판정 |
| 1~3 | 환경 준비, 영상 준비, 설정 |
| 4 | 영상 살펴보기와 기준선 위치 정하기 |
| 5 | **실험 1**: 모델·해상도별 작은 보행자 탐지 비교 |
| 6 | 탐지·추적·카운팅 파이프라인 |
| 7 | **실험 2**: 기본 버전 실행 |
| 8 | **실험 3**: 개선 버전 실행 (정지 객체 필터 + 클래스 다수결) |
| 9 | 결과 비교 표와 그래프 |
| 10~11 | 결과 영상, 보고서용 캡처 |
| 12 | **실험 4**: 속도 비교 (GPU) |
| 13 | 고찰, 제출 체크리스트, 참고문헌 |

### 시작 전에: GPU 켜기
메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 를 선택하고 저장하세요.
그다음 위에서부터 셀을 하나씩 실행(Shift + Enter)합니다.

## 0. 배경 이론

### 0.1 YOLO (You Only Look Once)
이미지를 **한 번만** 신경망에 통과시켜 객체의 위치(바운딩 박스)와 종류(클래스)를 동시에 예측하는 1단계(one-stage) 탐지기입니다.

| 단계 | 하는 일 |
| --- | --- |
| Backbone | 이미지에서 특징(모서리, 질감, 형태)을 뽑는다 |
| Neck | 크기가 다른 특징을 합쳐 작은 객체와 큰 객체를 모두 다룬다 |
| Head | 박스 좌표, 클래스, 신뢰도(confidence)를 출력한다 |
| NMS | 한 객체에 겹친 박스 중 가장 확실한 것만 남긴다 |

이번 과제는 COCO 데이터셋(80개 클래스)으로 미리 학습된 **YOLOv8**을 그대로 쓰고, 그중 교통 관련 5개 클래스만 고릅니다.

| COCO ID | 클래스 | 뜻 |
| --- | --- | --- |
| 0 | person | 보행자 |
| 2 | car | 승용차 |
| 3 | motorcycle | 오토바이 |
| 5 | bus | 버스 |
| 7 | truck | 트럭 |

### 0.2 ByteTrack (다중 객체 추적)
탐지는 프레임마다 따로 이루어지므로, "앞 프레임의 그 차"와 "지금 프레임의 이 차"가 같은지 알 수 없습니다.
ByteTrack은 박스의 위치와 겹침 정도로 앞뒤 프레임의 탐지를 이어 붙여 객체마다 **고유 ID** 를 붙입니다.
신뢰도가 낮은 탐지도 버리지 않고 기존 추적과 이어 보는 것이 특징이라, 잠깐 가려진 객체를 덜 놓칩니다.

### 0.3 기준선 통과 판정
점 $P$가 선분 $AB$의 어느 쪽에 있는지는 외적(cross product)의 부호로 알 수 있습니다.

$$
s(P) = (B_x - A_x)(P_y - A_y) - (B_y - A_y)(P_x - A_x)
$$

객체 중심이 직전 프레임 $P_{t-1}$에서 지금 $P_t$로 움직였을 때,

- $s(P_{t-1})$과 $s(P_t)$의 **부호가 다르고**,
- 이동 경로 $P_{t-1}P_t$의 양쪽에 $A$와 $B$가 있으면 (선분 범위 안에서 교차),

기준선을 통과한 것으로 보고, $s(P_t)$의 부호로 방향(+ / −)을 정합니다. 한 ID는 한 번만 셉니다.

### 0.4 세 가지 숫자
| 표시 | 뜻 | 특징 |
| --- | --- | --- |
| **now** | 지금 화면에 보이는 개수 | 순간 혼잡도 |
| **line** | 기준선을 통과한 개수 (방향별) | 교통량. 가장 믿을 만함 |
| **total** | 지금까지 나온 고유 추적 ID 수 | 가려짐·클래스 혼동으로 부풀기 쉬움 → 실험 3에서 개선 |

## 1. 환경 준비
### 1.1 GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("PyTorch", torch.__version__, "| GPU 사용 가능:", torch.cuda.is_available())

### 1.2 라이브러리 설치와 한글 글꼴
그래프에 한글이 깨지지 않도록 나눔고딕을 설치합니다.

In [ ]:
!pip install -q ultralytics yt-dlp
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import glob, os
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

def set_korean_font():
    candidates = glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf") + [r"C:\Windows\Fonts\malgun.ttf"]
    for path in candidates:
        if os.path.exists(path):
            fm.fontManager.addfont(path)
            mpl.rc("font", family=fm.FontProperties(fname=path).get_name())
            break
    mpl.rcParams["axes.unicode_minus"] = False

set_korean_font()
import ultralytics
ultralytics.checks()

## 2. 도로 영상 준비
두 방법 중 **하나만** 실행하세요. 결과는 `road.mp4` 로 저장됩니다.

**방법 A (권장): 내 PC의 영상 올리기**

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()                      # 파일 선택 창이 뜹니다
name = next(iter(uploaded))
shutil.move(name, "road.mp4")
print("저장됨: road.mp4", os.path.getsize("road.mp4") // 1024 // 1024, "MB")

**방법 B: YouTube에서 받기** - 수업 과제용으로만 쓰고 출처를 밝히세요.
Colab 서버는 YouTube가 막는 경우가 많습니다. 실패하면 방법 A를 쓰세요.

> 출처: Spotting Korea, 「삼성역 교차로 자동차 영상 4K Spotting Seoul」, YouTube

In [ ]:
URL = "https://www.youtube.com/watch?v=wVVlhfh724Q"
START, END = "00:30", "02:00"                  # 사용할 구간 (1분 30초)

!yt-dlp -q -f "bv*[height<=1080][ext=mp4]" -o road_full.mp4 "$URL"
!ffmpeg -y -loglevel error -ss $START -to $END -i road_full.mp4 -an -c copy road.mp4
!ls -lh road.mp4

## 3. 설정
모든 실험이 이 값을 씁니다. 바꾼 뒤에는 아래 셀들을 다시 실행하세요.

- `LINE` 은 화면 크기에 대한 비율 `(x1, y1, x2, y2)` 입니다 (0 = 왼쪽/위, 1 = 오른쪽/아래).
  세로선 `(0.45, 0.28, 0.45, 0.85)` 는 삼성역 영상에서 좌우로 지나는 차량과 아래쪽 횡단보도 보행자를 셉니다.
- `MAX_SECONDS` 를 `30` 처럼 주면 앞부분만 처리해 빠르게 시험할 수 있습니다. 전체는 `None`.

In [ ]:
SOURCE      = "road.mp4"
MODEL       = "yolov8s.pt"   # yolov8n(빠름) / yolov8s / yolov8m(정확)
IMGSZ       = 1280           # 작은 보행자는 1280 권장 (실험 1 참고)
CONF        = 0.4            # 차량 신뢰도 기준
PERSON_CONF = 0.25           # 보행자 신뢰도 기준 (작아서 낮게)
LINE        = (0.45, 0.28, 0.45, 0.85)
MAX_SECONDS = None           # 앞부분만: 숫자(초) / 전체: None

# 개선 버전(실험 3) 설정
STATIC_SECONDS = 20          # 이만큼 지난 뒤부터 정지 물체를 판단 (초반엔 지나가는 보행자도 비율이 높게 나옴)
STATIC_CELL    = 32          # 화면을 나누는 칸 크기 (1080p 기준 픽셀)
STATIC_RATIO   = 0.15        # 지금까지 프레임의 15% 이상 '보행자'가 나온 칸 = 신호등 같은 고정 물체
MIN_TRACK      = 5           # 이보다 짧게(프레임) 보인 추적 ID는 total에서 제외

## 4. 영상 살펴보기와 기준선 위치 정하기
영상 정보를 확인하고, 처음·중간·끝 장면에 기준선을 그려 봅니다. 위치가 어색하면 3번의 `LINE` 을 고치고 다시 실행하세요.

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture(SOURCE)
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS_IN, N_FRAMES = cap.get(cv2.CAP_PROP_FPS) or 30, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
assert W > 0, "영상을 읽을 수 없습니다. 2번 단계를 확인하세요."
print(f"영상: {W}x{H}, {FPS_IN:.1f} fps, {N_FRAMES} 프레임 ({N_FRAMES / FPS_IN:.0f}초)")

A = (int(LINE[0] * W), int(LINE[1] * H))
B = (int(LINE[2] * W), int(LINE[3] * H))
SCALE = W / 1920                                # 픽셀 기준값을 해상도에 맞추는 비율

fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))
for ax, r in zip(axes, (0.0, 0.5, 0.95)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(N_FRAMES * r))
    ok, frame = cap.read()
    cv2.line(frame, A, B, (0, 255, 255), 6)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); ax.set_title(f"{N_FRAMES * r / FPS_IN:.0f}초"); ax.axis("off")
cap.release()
plt.suptitle("기준선(노란 선) 위치 확인"); plt.tight_layout(); plt.show()

## 5. 실험 1: 모델·해상도별 작은 보행자 탐지 비교
높은 곳에서 찍은 영상은 사람이 수십 픽셀 크기로 작게 나옵니다.
같은 장면 한 장을 모델 크기(n/s/m)와 입력 해상도(640/1280)를 바꿔 가며 탐지해,
**몇 명·몇 대가 잡히는지** 와 **얼마나 걸리는지** 를 비교합니다.

> 가설: 해상도를 키우면 작은 보행자가 더 잘 잡히지만 느려진다.

In [ ]:
import time
import pandas as pd
from ultralytics import YOLO

TARGET_CLASSES = {0: "Person", 2: "Car", 3: "Motorcycle", 5: "Bus", 7: "Truck"}
VEHICLE_IDS = {2, 3, 5, 7}
device = 0 if torch.cuda.is_available() else "cpu"

cap = cv2.VideoCapture(SOURCE)
cap.set(cv2.CAP_PROP_POS_FRAMES, min(600, N_FRAMES // 3))
_, sample = cap.read()
cap.release()

exp1, plots = [], {}
for m in ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]:
    mdl = YOLO(m)
    for sz in (640, 1280):
        for _ in range(3):                                                   # 워밍업 (첫 실행은 느림)
            mdl(sample, imgsz=sz, device=device, verbose=False)
        t = time.time()
        r = mdl(sample, classes=list(TARGET_CLASSES), conf=PERSON_CONF, imgsz=sz, device=device, verbose=False)[0]
        ms = (time.time() - t) * 1000
        cls, conf = r.boxes.cls.int().tolist(), r.boxes.conf.tolist()
        persons = sum(1 for c, p in zip(cls, conf) if c == 0 and p >= PERSON_CONF)
        vehicles = sum(1 for c, p in zip(cls, conf) if c in VEHICLE_IDS and p >= CONF)
        exp1.append({"모델": m.replace(".pt", ""), "해상도": sz, "보행자": persons, "차량": vehicles, "시간(ms)": round(ms)})
        plots[(m, sz)] = r.plot(line_width=2, font_size=10)

exp1 = pd.DataFrame(exp1)
exp1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
labels = exp1["모델"] + "\n" + exp1["해상도"].astype(str)
colors = ["#9db4d6" if s == 640 else "#2f5d9e" for s in exp1["해상도"]]
for ax, col, title in zip(axes, ["보행자", "차량", "시간(ms)"], ["잡힌 보행자 수", "잡힌 차량 수", "한 장 처리 시간 (ms)"]):
    bars = ax.bar(labels, exp1[col], color=colors)
    ax.bar_label(bars)
    ax.set_title(title)
    ax.yaxis.get_major_locator().set_params(integer=True)
    ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("실험 1: 같은 장면, 모델·해상도별 결과 (연한색 640 / 진한색 1280)")
plt.tight_layout(); plt.show()

작은 보행자가 몰려 있는 아래쪽 횡단보도 부분을 확대해 가장 가벼운 설정과 이번 시스템 설정을 나란히 봅니다.

In [ ]:
y0, y1, x0, x1 = int(0.62 * H), int(0.95 * H), int(0.05 * W), int(0.70 * W)   # 확대할 영역
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
for ax, key, title in zip(axes, [("yolov8n.pt", 640), (MODEL, IMGSZ)],
                          ["가벼운 설정: yolov8n · 640", f"이번 시스템: {MODEL[:-3]} · {IMGSZ}"]):
    ax.imshow(cv2.cvtColor(plots[key][y0:y1, x0:x1], cv2.COLOR_BGR2RGB)); ax.set_title(title, fontsize=14); ax.axis("off")
plt.tight_layout(); plt.show()

**실험 1 정리 (직접 적어 보세요)**
- 보행자 수가 가장 많이 잡힌 설정은 ___ 이고, 640 → 1280 으로 바꾸면 보행자가 ___ 명 → ___ 명으로 바뀌었다.
- 대신 처리 시간은 ___ ms → ___ ms 로 늘었다. 이 결과로 이번 시스템은 `MODEL`, `IMGSZ` 를 ___ 로 정했다.

## 6. 탐지·추적·카운팅 파이프라인
`run_pipeline()` 하나로 영상 전체를 처리합니다. `improved=True` 이면 실험 3의 개선 기능 세 가지가 켜집니다.

| 개선 기능 | 해결하려는 문제 | 방법 |
| --- | --- | --- |
| 정지 객체 필터 | 신호등·기둥이 보행자로 오탐지 | 화면을 `STATIC_CELL` 크기 칸으로 나눠, 전체 프레임의 `STATIC_RATIO` 이상 '보행자'가 나온 칸의 보행자는 제외 (회색 박스). 신호등은 탐지가 끊겼다 이어지며 ID가 계속 바뀌므로, ID가 아니라 **위치**로 판단 |
| 클래스 다수결 | 같은 차가 트럭↔버스로 바뀜 | 추적 ID별로 지금까지 가장 많이 나온 클래스를 사용 |
| 짧은 추적 제거 | 잠깐 나타난 ID 때문에 total 부풀림 | `MIN_TRACK` 프레임보다 짧은 ID는 total에서 제외 |

In [ ]:
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from IPython.display import display, clear_output
from PIL import Image

COLORS = {0: (0, 200, 255), 2: (255, 120, 0), 3: (200, 0, 200), 5: (0, 200, 0), 7: (0, 0, 220)}   # BGR
LINE_COLOR, FLASH_COLOR, STATIC_COLOR = (0, 255, 255), (0, 255, 0), (150, 150, 150)


def side(p, a, b):
    """점 p가 선분 a-b의 어느 쪽에 있는지 (+/-/0): 0.3절의 s(P)"""
    return (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0])


def crossed(p_prev, p_now, a, b):
    """이동 경로(p_prev -> p_now)가 기준선 a-b를 가로질렀으면 방향(+1/-1), 아니면 0"""
    s1, s2 = side(p_prev, a, b), side(p_now, a, b)
    if s1 == 0 or s1 * s2 > 0:
        return 0
    if side(a, p_prev, p_now) * side(b, p_prev, p_now) > 0:   # 선분 범위 밖
        return 0
    return 1 if s2 > 0 else -1


def draw_panel(frame, current, line_count, totals, fps, tag):
    """왼쪽 위에 now / line / total 표와 FPS를 그린다. totals = TotalCounter"""
    passed = lambda cids: sum(line_count[c][1] + line_count[c][-1] for c in cids)
    rows = [("", "now", "line", "total")]
    groups = [("Vehicles", VEHICLE_IDS), ("Persons", [0])] + [("  " + TARGET_CLASSES[c], [c]) for c in (2, 3, 5, 7)]
    for name, cids in groups:
        rows.append((name, str(sum(current[c] for c in cids)), str(passed(cids)), str(sum(totals.get(c) for c in cids))))
    h = 30 * (len(rows) + 1) + 16
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (610, 10 + h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
    cv2.putText(frame, f"FPS: {fps:.1f}  [{tag}]", (22, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
    for i, cells in enumerate(rows):
        color = (180, 180, 180) if i == 0 else (255, 255, 255)
        for x, text in zip([22, 260, 380, 500], cells):
            cv2.putText(frame, text, (x, 72 + 30 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)


class TotalCounter:
    """클래스별 total(누적 고유 ID 수)을 관리한다.

    기본 버전: 탐지된 클래스마다 ID를 따로 센다 (같은 ID가 트럭·버스 양쪽에 들어갈 수 있음).
    개선 버전: ID마다 다수결 클래스 하나로 세고, 짧은 추적과 정지 물체는 뺀다.
               바뀐 ID만 다시 반영해서 매 프레임 전체를 다시 세지 않는다.
    """

    def __init__(self, improved, min_track):
        self.improved, self.min_track = improved, min_track
        self.votes = defaultdict(Counter)   # ID별 클래스 투표
        self.seen = Counter()               # ID별 등장 프레임 수
        self.static = set()                 # 정지 물체로 판정된 ID
        self.raw = defaultdict(set)         # 기본 버전용: 클래스별 ID
        self.state = {}                     # 개선 버전용: ID -> 지금 total 에 들어가 있는 클래스
        self.counts = Counter()             # 개선 버전용: 클래스별 total

    def voted(self, tid):
        return self.votes[tid].most_common(1)[0][0]

    def _refresh(self, tid):
        new = self.voted(tid) if self.seen[tid] >= self.min_track and tid not in self.static else None
        old = self.state.get(tid)
        if new != old:
            if old is not None:
                self.counts[old] -= 1
            if new is not None:
                self.counts[new] += 1
            self.state[tid] = new

    def add(self, tid, cls):
        self.raw[cls].add(tid)
        self.votes[tid][cls] += 1
        self.seen[tid] += 1
        if self.improved:
            self._refresh(tid)

    def mark_static(self, tid):
        if tid not in self.static:
            self.static.add(tid)
            self._refresh(tid)

    def get(self, cls):
        return self.counts[cls] if self.improved else len(self.raw[cls])


def run_pipeline(improved=False, save=None, max_seconds=MAX_SECONDS, preview_every=10):
    """영상을 처리하고 결과(dict)를 돌려준다. improved=True 이면 개선 기능 사용."""
    tag = "improved" if improved else "baseline"
    model = YOLO(MODEL)                                         # 실행마다 새 모델 = 추적 상태 초기화
    cap = cv2.VideoCapture(SOURCE)
    writer = cv2.VideoWriter(save, cv2.VideoWriter_fourcc(*"mp4v"), FPS_IN, (W, H)) if save else None
    limit = min(int(max_seconds * FPS_IN), N_FRAMES) if max_seconds else N_FRAMES
    static_frames, cell = int(STATIC_SECONDS * FPS_IN), max(int(STATIC_CELL * SCALE), 1)

    last_pos, counted = {}, set()
    line_count = defaultdict(lambda: {1: 0, -1: 0})
    totals = TotalCounter(improved, MIN_TRACK)                  # total (누적 고유 ID)
    heat = Counter()                                            # 칸별 '보행자'가 나온 프레임 수 (정지 판정용)
    series = []                                                 # 프레임별 기록 (그래프용)
    flash, fps, prev, start = 0, 0.0, time.time(), time.time()

    for idx in tqdm(range(limit), desc=tag):
        ok, frame = cap.read()
        if not ok:
            break
        res = model.track(frame, classes=list(TARGET_CLASSES), conf=min(CONF, PERSON_CONF), imgsz=IMGSZ,
                          device=device, persist=True, tracker="bytetrack.yaml", verbose=False)[0]
        boxes = res.boxes
        current, static_now = defaultdict(int), 0
        person_cells = set()

        if boxes is not None and len(boxes):
            ids = boxes.id.int().tolist() if boxes.id is not None else [None] * len(boxes)
            for xyxy, cls, conf, tid in zip(boxes.xyxy.tolist(), boxes.cls.int().tolist(), boxes.conf.tolist(), ids):
                if conf < (PERSON_CONF if cls == 0 else CONF):  # 클래스별 신뢰도 기준
                    continue
                x1, y1, x2, y2 = map(int, xyxy)
                center = ((x1 + x2) // 2, (y1 + y2) // 2)
                is_static = False
                if tid is not None:
                    totals.add(tid, cls)
                    if improved:
                        cls = totals.voted(tid)                                 # 클래스 다수결
                if cls == 0:
                    gx, gy = center[0] // cell, center[1] // cell
                    person_cells.add((gx, gy))
                    if improved and idx >= static_frames:                       # 주변 3x3 칸 중 가장 자주 나온 비율
                        ratio = max(heat[(gx + dx, gy + dy)] for dx in (-1, 0, 1) for dy in (-1, 0, 1)) / idx
                        is_static = ratio >= STATIC_RATIO
                        if is_static and tid is not None:
                            totals.mark_static(tid)
                if is_static:                                                   # 정지 객체: 세지 않고 회색 표시
                    static_now += 1
                    cv2.rectangle(frame, (x1, y1), (x2, y2), STATIC_COLOR, 1)
                    continue

                current[cls] += 1                                               # now
                if tid is not None:
                    if tid in last_pos and tid not in counted:
                        d = crossed(last_pos[tid], center, A, B)
                        if d:
                            line_count[cls][d] += 1                             # line
                            counted.add(tid)
                            flash = 6
                    last_pos[tid] = center

                color = COLORS[cls]
                label = f"{TARGET_CLASSES[cls]} {conf:.2f}" + (f" #{tid}" if tid is not None else "")
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.circle(frame, center, 3, color, -1)
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 6, y1), color, -1)
                cv2.putText(frame, label, (x1 + 3, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

        heat.update(person_cells)                                       # 칸별 보행자 등장 횟수 누적


        cv2.line(frame, A, B, FLASH_COLOR if flash else LINE_COLOR, 4, cv2.LINE_AA)
        cv2.putText(frame, "COUNT LINE", (A[0] + 10, A[1] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.8, LINE_COLOR, 2, cv2.LINE_AA)
        flash = max(flash - 1, 0)
        now = time.time()
        inst = 1.0 / max(now - prev, 1e-6)
        fps, prev = (0.9 * fps + 0.1 * inst if fps else inst), now
        draw_panel(frame, current, line_count, totals, fps, tag)

        series.append({"t": idx / FPS_IN,
                       "now_vehicle": sum(current[c] for c in VEHICLE_IDS), "now_person": current[0],
                       "line_vehicle": sum(line_count[c][1] + line_count[c][-1] for c in VEHICLE_IDS),
                       "line_person": line_count[0][1] + line_count[0][-1], "static": static_now})
        if writer:
            writer.write(frame)
        if preview_every and idx % int(FPS_IN * preview_every) == 0:
            clear_output(wait=True)
            display(Image.fromarray(cv2.cvtColor(cv2.resize(frame, (960, 540)), cv2.COLOR_BGR2RGB)))

    cap.release()
    if writer:
        writer.release()
    elapsed = time.time() - start
    print(f"[{tag}] {idx + 1} 프레임, {elapsed:.0f}초 (평균 {(idx + 1) / elapsed:.1f} FPS)")
    return {"tag": tag, "line": {c: dict(v) for c, v in line_count.items()}, "total": {c: totals.get(c) for c in TARGET_CLASSES}, "static_ids": len(totals.static),
            "series": pd.DataFrame(series), "fps": (idx + 1) / elapsed, "frames": idx + 1}

## 7. 실험 2: 기본 버전 실행
개선 기능 없이 그대로 돌립니다. 진행 막대와 함께 10초 분량마다 현재 화면을 보여 줍니다.

> Colab에는 `cv2.imshow` 창이 없어서 미리 보기를 셀 아래에 표시하고 결과는 영상 파일로 저장합니다.

In [ ]:
base = run_pipeline(improved=False, save="result_baseline.mp4")

## 8. 실험 3: 개선 버전 실행
같은 영상을 개선 기능(정지 객체 필터, 클래스 다수결, 짧은 추적 제거)을 켜고 다시 처리합니다.
회색 얇은 박스가 "움직이지 않아서 보행자에서 뺀 물체"입니다.

In [ ]:
impr = run_pipeline(improved=True, save="result.mp4")

## 9. 결과 비교
### 9.1 클래스별 집계 표

In [ ]:
def summary(r):
    rows = {}
    for cid, name in TARGET_CLASSES.items():
        c = r["line"].get(cid, {})
        rows[name] = {"line +": c.get(1, 0), "line -": c.get(-1, 0),
                      "line 합계": c.get(1, 0) + c.get(-1, 0), "total": r["total"].get(cid, 0)}
    df = pd.DataFrame(rows).T
    df.loc["차량 합계"] = df.loc[["Car", "Motorcycle", "Bus", "Truck"]].sum()
    return df

compare = pd.concat({"기본": summary(base), "개선": summary(impr)}, axis=1)
compare

### 9.2 기본 vs 개선: total이 얼마나 줄었나
line은 두 버전이 비슷해야 정상이고(교통량은 그대로), total은 부풀린 부분이 줄어야 합니다.

In [ ]:
names = list(TARGET_CLASSES.values())
x = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(18, 4.8))
for ax, key, title in zip(axes, ["total", "line 합계"], ["total (고유 ID 수)", "line (기준선 통과 수)"]):
    b1 = ax.bar(x - 0.2, compare[("기본", key)][names], 0.4, label="기본", color="#b8c2cc")
    b2 = ax.bar(x + 0.2, compare[("개선", key)][names], 0.4, label="개선", color="#2f5d9e")
    ax.bar_label(b1); ax.bar_label(b2)
    ax.set_xticks(x, names); ax.set_title(title); ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("실험 2 vs 실험 3"); plt.tight_layout(); plt.show()

### 9.3 시간에 따른 변화 (개선 버전)

In [ ]:
s = impr["series"].copy()
s["sec"] = s["t"].astype(int)
per_sec = s.groupby("sec").agg(now_vehicle=("now_vehicle", "mean"), now_person=("now_person", "mean"),
                               line_vehicle=("line_vehicle", "max"), line_person=("line_person", "max"))

fig, axes = plt.subplots(1, 2, figsize=(18, 4.8))
axes[0].plot(per_sec.index, per_sec["now_vehicle"], color="#2f5d9e", label="차량")
axes[0].plot(per_sec.index, per_sec["now_person"], color="#e07b00", label="보행자")
axes[0].set_title("화면 속 객체 수 (now, 초 단위 평균)")
axes[1].plot(per_sec.index, per_sec["line_vehicle"], color="#2f5d9e", label="차량")
axes[1].plot(per_sec.index, per_sec["line_person"], color="#e07b00", label="보행자")
axes[1].set_title("기준선 누적 통과 수 (line)")
for ax in axes:
    ax.set_xlabel("시간 (초)"); ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

dur_min = impr["frames"] / FPS_IN / 60
veh = per_sec["line_vehicle"].iloc[-1]
print(f"처리한 길이 {dur_min * 60:.0f}초 동안 기준선 통과 차량 {veh}대 → 시간당 약 {veh / dur_min * 60:,.0f}대로 환산")

### 9.4 방향별 통과 수

In [ ]:
lc = summary(impr).loc[names, ["line +", "line -"]]
ax = lc.plot.barh(figsize=(10, 4), color=["#2f5d9e", "#9db4d6"])
for c in ax.containers:
    ax.bar_label(c)
ax.set_title("기준선 방향별 통과 수 (+ / -)"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 10. 결과 영상 보기·내려받기
OpenCV로 저장한 영상은 브라우저에서 바로 재생되지 않아서 H.264로 변환합니다. 미리 보기는 크기를 줄인 앞 30초입니다.

In [ ]:
from base64 import b64encode
from IPython.display import HTML

!ffmpeg -y -loglevel error -i result.mp4 -c:v libx264 -pix_fmt yuv420p -crf 23 result_h264.mp4
!ffmpeg -y -loglevel error -i result_h264.mp4 -t 30 -vf scale=960:-2 -c:v libx264 -crf 28 preview.mp4

data = b64encode(open("preview.mp4", "rb").read()).decode()
HTML(f'<video width=960 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

In [ ]:
from google.colab import files
files.download("result_h264.mp4")   # 개선 버전 전체 영상

## 11. 보고서용 캡처
같은 순간을 기본·개선 버전으로 나란히 저장합니다. 회색 박스(정지 물체 제외)와 패널 숫자 차이를 보고서에 쓰기 좋습니다.

In [ ]:
from google.colab import files
os.makedirs("captures", exist_ok=True)
caps = [cv2.VideoCapture("result_baseline.mp4"), cv2.VideoCapture("result.mp4")]
n = int(caps[1].get(cv2.CAP_PROP_FRAME_COUNT))
picks = [int(n * r) for r in (0.3, 0.95)]
fig, axes = plt.subplots(len(picks), 2, figsize=(20, 6 * len(picks)))
for row, f in enumerate(picks):
    for col, (cap, name) in enumerate(zip(caps, ["기본", "개선"])):
        cap.set(cv2.CAP_PROP_POS_FRAMES, f)
        ok, img = cap.read()
        if ok:
            cv2.imwrite(f"captures/{['baseline', 'improved'][col]}_{f}.jpg", img)
            axes[row, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row, col].set_title(f"{name} · {f / FPS_IN:.0f}초", fontsize=14); axes[row, col].axis("off")
for cap in caps:
    cap.release()
plt.tight_layout(); plt.show()
!zip -q -r captures.zip captures
files.download("captures.zip")

## 12. 실험 4: 속도 비교
모델·해상도별로 탐지만의 속도(FPS)를 잽니다. 추적·그리기·저장은 포함하지 않습니다.
실시간의 기준은 보통 **30 FPS** 입니다.

In [ ]:
def measure_fps(mdl, frames, imgsz):
    mdl(frames[0], classes=list(TARGET_CLASSES), imgsz=imgsz, device=device, verbose=False)   # 워밍업
    t = time.time()
    for f in frames:
        mdl(f, classes=list(TARGET_CLASSES), imgsz=imgsz, device=device, verbose=False)
    return len(frames) / (time.time() - t)

cap = cv2.VideoCapture(SOURCE)
frames = [cap.read()[1] for _ in range(150)]
cap.release()

bench = []
for m in ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]:
    mdl = YOLO(m)
    for sz in (640, 1280):
        bench.append({"모델": m[:-3], "해상도": sz, "FPS": round(measure_fps(mdl, frames, sz), 1)})
bench = pd.DataFrame(bench)

pivot = bench.pivot(index="모델", columns="해상도", values="FPS")
ax = pivot.plot.bar(figsize=(10, 4.5), color=["#9db4d6", "#2f5d9e"], rot=0)
for c in ax.containers:
    ax.bar_label(c)
ax.axhline(30, color="#c0392b", linestyle="--", linewidth=1)
ax.text(2.45, 31, "실시간 30 FPS", color="#c0392b", ha="right")
ax.set_title(f"탐지 속도 ({torch.cuda.get_device_name(0) if device == 0 else 'CPU'})")
ax.set_ylabel("FPS"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
pivot

## 13. 고찰

### 13.1 생각해 볼 질문 (보고서에 답을 적어 보세요)
1. 실험 1에서 해상도를 640 → 1280 으로 올렸을 때 보행자 수와 처리 시간은 각각 어떻게 변했나? 왜 그럴까?
2. 실험 2에서 total이 line보다 훨씬 큰 이유는 무엇인가? (힌트: 가려짐, 클래스 바뀜, 짧게 끊긴 추적)
3. 실험 3에서 정지 물체로 판정된 ID는 몇 개인가(`impr["static_ids"]`)? 이 필터가 **한자리에 오래 서 있는 진짜 보행자**(신호 대기)까지 빼 버릴 위험은 없는가?
4. 기준선 위치를 가로선으로 바꾸면 무엇을 세게 되는가? 이 교차로에서 교통량을 가장 잘 세려면 기준선을 몇 개, 어디에 두어야 할까?
5. 실험 4에서 실시간(30 FPS)을 넘는 설정은 무엇인가? 정확도와 속도 중 무엇을 택할지는 어떤 상황에 따라 달라지나?

### 13.2 한계와 향후 과제
- **작은 객체**: 높은 시점의 보행자는 1280 해상도로도 일부를 놓친다 → 화면을 나눠 탐지(타일링, SAHI), 높은 시점 데이터로 추가 학습
- **오탐지**: 위치 기반 정지 필터는 규칙 기반이라, 같은 자리에 오래 서 있는 보행자도 걸러질 수 있다 → 신호등 영역 지정, 보행자 전용 모델
- **국내 환경**: 한국 버스·트럭 모양에 맞춘 AI Hub 데이터로 fine-tuning
- **활용**: 차량과 보행자가 가까워지면 경고하는 위험 감지 기능

### 13.3 제출 체크리스트
- [ ] 1~12번 셀을 모두 실행했고 오류가 없다
- [ ] 실험 1 정리 칸과 13.1 질문에 답을 적었다
- [ ] 9번 비교 표·그래프와 11번 캡처를 보고서에 넣었다
- [ ] 결과 영상(`result_h264.mp4`)을 내려받았다
- [ ] 영상 출처를 보고서에 적었다
- [ ] **파일 → 다운로드 → .ipynb** 로 실행 결과가 담긴 노트북을 저장했다

### 참고문헌
1. Redmon, J. et al. (2016). [You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640). CVPR.
2. Jocher, G. et al. (2023). [Ultralytics YOLOv8](https://docs.ultralytics.com/models/yolov8/).
3. Zhang, Y. et al. (2022). [ByteTrack: Multi-Object Tracking by Associating Every Detection Box](https://arxiv.org/abs/2110.06864). ECCV.
4. Lin, T.-Y. et al. (2014). [Microsoft COCO: Common Objects in Context](https://arxiv.org/abs/1405.0312). ECCV.
5. Spotting Korea. [삼성역 교차로 자동차 영상 4K Spotting Seoul](https://www.youtube.com/watch?v=wVVlhfh724Q). YouTube. (테스트 영상)